<a href="https://colab.research.google.com/github/LetheeshBalaji/DEEP-LEARNING/blob/main/Exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PART A - PRE-TRAINED LLM
# Register Number: YOUR_ROLL_NUMBER

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load pre-trained model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Sample user prompt
prompt = "Artificial Intelligence is"

# Tokenization
inputs = tokenizer(prompt, return_tensors="pt")

print("Tokenized Input:", inputs["input_ids"])

# Generate response
with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        max_length=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode generated response
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nGenerated Response:")
print(response)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenized Input: tensor([[8001, 9542, 9345,  318]])

Generated Response:
Artificial Intelligence is a tool of intelligence developed by researchers at Johns Hopkins University (JHU) to develop artificial intelligence, which is designed for the search for information from the Internet. The software enables humans to search for information from the Web, to gather information from the Internet.














































In [ ]:
# PART B - CHATBOT FUNCTIONALITY
# Register Number: YOUR_ROLL_NUMBER

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Conversation history
chat_history = []

# Chatbot function
def chatbot(user_query):

    # Construct prompt using conversation history
    context = ""

    for question, answer in chat_history[-3:]:
        context += f"User: {question}\nBot: {answer}\n"

    prompt = context + f"User: {user_query}\nBot:"

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=400
    )

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    # Save conversation
    chat_history.append((user_query, response))

    return response


# Test chatbot
print("LLM CHATBOT")
print("Type exit to stop")

while True:
    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    answer = chatbot(query)
    print("Bot:", answer)h

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

LLM CHATBOT
Type exit to stop

You: hello
Bot:  http://www.youtube.com/watch?v=kF5dSkFx7Xy
http://www.youtube.com/watch?v=Nx8rPfW3z3
http://www.youtube.com/watch?v=l

You: exit


In [6]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [7]:
# PART C - STREAMLIT CHATBOT INTERFACE
# Register Number: YOUR_ROLL_NUMBER

import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model
@st.cache_resource
def load_model():

    model_name = "distilgpt2"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    return tokenizer, model


tokenizer, model = load_model()

# Page configuration
st.set_page_config(page_title="LLM Chatbot")

st.title("Pre-trained LLM Chatbot")
st.write("Chat with an AI-powered language model.")

# Store chat history
if "messages" not in st.session_state:
    st.session_state.messages = []


# Generate response
def generate_response(query):

    prompt = f"User: {query}\nBot:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response


# Display previous messages
for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])


# User input
user_input = st.chat_input("Enter your message...")

if user_input:

    # Display user message
    st.session_state.messages.append(
        {"role": "user", "content": user_input}
    )

    with st.chat_message("user"):
        st.write(user_input)

    # Generate and display response
    with st.chat_message("assistant"):

        with st.spinner("Generating response..."):
            response = generate_response(user_input)

        st.write(response)

    st.session_state.messages.append(
        {"role": "assistant", "content": response}
    )

2026-11-05 19:45:52.435 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

2026-11-05 19:45:53.828 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 19:45:53.831 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 19:45:54.012 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-11-05 19:45:54.013 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 19:45:54.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 19:45:54.015 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 19:45:54.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn